In [1]:
import pystac
from datetime import datetime
import rasterio
import rasterio.warp

In [5]:
import sys
from pathlib import Path
# Add the src directory to sys.path
sys.path.append(str(Path().resolve().parents[0] / "src"))

In [6]:
from datasource.chelsa import s3

# Select a file from the S3 chelsa repository

In [16]:
test_url = s3.format_url_month_ts("pr", 1, 2000)
print(test_url)  

https://os.unil.cloud.switch.ch/chelsa02/chelsa/global/monthly/pr/2000/CHELSA_pr_01_2000_V.2.1.tif


In [17]:
src.bounds

BoundingBox(left=-180.00013888885002, bottom=-90.00013888885, right=179.99985967115003, top=83.99986041515)

In [24]:
def generate_stac_item(url, stac_id, dt, band_id, band_title, output_path):
    if isinstance(dt, str):
        # Parses "YYYY-MM-DD" strings into datetime objects
        dt = datetime.strptime(dt, "%Y-%m-%d")
    with rasterio.open(url) as src:
        # --- A. Automate Media Type using Rasterio Driver ---
        driver = src.driver
        media_type = "application/octet-stream" # Default
        if driver == 'GTiff':
            # Check if it is likely Cloud Optimized (Tiled + Compressed)
            # This distinguishes a basic TIFF from a COG
            if src.profile.get('tiled') and src.profile.get('compress'):
                 media_type = pystac.MediaType.COG
            else:
                 media_type = pystac.MediaType.GEOTIFF          
        elif driver in ['JP2OpenJPEG', 'JPEG2000']:
            media_type = pystac.MediaType.JPEG2000
        elif driver == 'netCDF':
            media_type = "application/x-netcdf"   
        elif driver in ['HDF4', 'HDF5', 'HDF4Image', 'HDF5Image']:
            media_type = "application/x-hdf"     
        elif driver == 'PNG':
            media_type = pystac.MediaType.PNG    
        elif driver == 'JPEG':
            media_type = pystac.MediaType.JPEG
        """
        STAC catalogue are searchable and discoverable spatially speaking in terms of longitude and latitude
            - proj_bounds saves the coordinates of the file in its native CRS
            - left, bottom, right, top saves the coordinates in lat/lon for searchability and discoverability
        """
        proj_bounds = list(src.bounds)
        #Transform bounds from src_crs to dst_crs (4326).
        left, bottom, right, top = rasterio.warp.transform_bounds(src.crs, "EPSG:4326", *src.bounds)
        item = pystac.Item(id=stac_id,
                           geometry={"type":"Polygon",
                                     "coordinates": [[[left, bottom],
                                                    [right, bottom],
                                                    [right, top],
                                                    [left, top],
                                                    [left, bottom]
                                                ]]},
                           bbox=[left, bottom, right, top],
                           datetime=dt,
                           # Caveat ("proj:shape"): this is [height, width] and not [width, height] if you want to set them yourself
                           properties={"proj:epsg": src.crs.to_epsg(),
                                       "proj:shape": src.shape,
                                       "proj:bbox": proj_bounds,},
                           stac_extensions=["https://stac-extensions.github.io/eo/v1.1.0/schema.json",
                                            "https://stac-extensions.github.io/projection/v1.1.0/schema.json",],
                           assets={band_id: pystac.Asset(href=url,
                                                         title=band_title,
                                                         media_type=media_type,
                                                         roles=["data"],
                                                         # REQUIRED: define the bands in the eo extension for openEO to be able to load it
                                                         extra_fields={"eo:bands": [{"name": band_id,}],})})
        item.validate()
        #Include self link will reference absolute local path. We need it to be discoverable for the backend
        item.save_object(dest_href=output_path, include_self_link=False)

In [25]:
generate_stac_item(test_url, "CHELSAmonthly_pr_01012000", "2000-01-10", "pr", "pr 01/01/2000 CHELSA", "testSTAC.json")

In [ ]:
with rasterio.open(tiff_url) as src:
    proj_bounds = list(src.bounds)
    left, bottom, right, top = rasterio.warp.transform_bounds(src.crs, "EPSG:4326", *src.bounds)
    item = pystac.Item(
        id="ndvi-example-stac-item",
        geometry={
            "type": "Polygon",
            "coordinates": [[
                [left, bottom],
                [right, bottom],
                [right, top],
                [left, top],
                [left, bottom]
            ]]
        },
        bbox=[left, bottom, right, top],
        datetime= dt,
        properties={ # These properties are optional, but can speed up the loading of the data.
            "proj:epsg": src.crs.to_epsg(),
            "proj:shape": src.shape, # Caveat: this is [height, width] and not [width, height] if you want to set them yourself
            "proj:bbox": proj_bounds,
        },
        stac_extensions=[
            "https://stac-extensions.github.io/eo/v1.1.0/schema.json",
            "https://stac-extensions.github.io/projection/v1.1.0/schema.json",
        ],
        assets={
            "ndvi": pystac.Asset(
                href=tiff_url,
                title="Normalized Difference Vegetation Index",
                extra_fields={
                    "eo:bands": [ # REQUIRED: define the bands in the eo extension for openEO to be able to load it
                        {
                            "name": "NDVI-band",
                        }
                    ],
                }
            )
        }
    )
    item.validate()
    item.save_object(dest_href=output_path, include_self_link=False)

In [ ]:
### Multi band tiff file

# Inside your pystac.Item definition...
assets={
    "visual": pystac.Asset(  # One key for the whole file
        href=tiff_url,
        title="RGB Image",
        roles=["data", "visual"],
        extra_fields={
            "eo:bands": [
                {
                    "name": "red",          # Band 1 in the Tiff
                    "common_name": "red",   # Optional: Helps clients find "red" automatically
                    "description": "Band 1 - Red Channel"
                },
                {
                    "name": "green",        # Band 2 in the Tiff
                    "common_name": "green",
                    "description": "Band 2 - Green Channel"
                },
                {
                    "name": "blue",         # Band 3 in the Tiff
                    "common_name": "blue",
                    "description": "Band 3 - Blue Channel"
                }
            ]
        }
    )
}

### Multiband multi tiff files

# distinct file paths
red_url = "path/to/image_red.tif"
nir_url = "path/to/image_nir.tif"

assets={
    "red": pystac.Asset(
        href=red_url,
        title="Red Band",
        roles=["data"],
        extra_fields={
            "eo:bands": [{
                "name": "B04",          # The specific ID
                "common_name": "red"    # The standard alias
            }]
        }
    ),
    "nir": pystac.Asset(
        href=nir_url,
        title="NIR Band",
        roles=["data"],
        extra_fields={
            "eo:bands": [{
                "name": "B08",
                "common_name": "nir"
            }]
        }
    )
}

### Tiled tiff files

import pystac
from pystac import CatalogType
import rasterio
from rasterio.warp import transform_bounds
from datetime import datetime
import os

# 1. Create the Collection (The Dataset Container)
collection = pystac.Collection(
    id="clc-plus-timeseries",
    title="Corine Land Cover Plus - Time Series",
    description="Monthly Land Cover classification from 2018-2020.",
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([[-180, -90, 180, 90]]),  # Placeholder
        temporal=pystac.TemporalExtent([[datetime.now(), datetime.now()]]) # Placeholder
    ),
    stac_extensions=[
        "https://stac-extensions.github.io/eo/v1.1.0/schema.json",
        "https://stac-extensions.github.io/projection/v1.1.0/schema.json"
    ]
)

# Imagine a list of file paths (could be different times OR different tiles)
file_list = [
    "data/CLC_2018_01_Tile1.tif",
    "data/CLC_2018_02_Tile1.tif", 
    "data/CLC_2018_01_Tile2.tif"
]

items = []

for file_path in file_list:
    with rasterio.open(file_path) as src:
        # --- A. Spatial Extraction (Handles Tiles) ---
        # Each tile gets its own unique BBOX
        proj_bounds = list(src.bounds)
        proj_epsg = src.crs.to_epsg()
        left, bottom, right, top = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
        bbox = [left, bottom, right, top]
        
        geometry = {
            "type": "Polygon",
            "coordinates": [[[left, bottom], [right, bottom], [right, top], [left, top], [left, bottom]]]
        }

        # --- B. Temporal Extraction (Handles Time Steps) ---
        # Strategy 1: Parse filename (e.g., 'CLC_2018_01...')
        # date_str = os.path.basename(file_path).split('_')[1:3] 
        # item_date = datetime(int(date_str[0]), int(date_str[1]), 1)
        
        # Strategy 2: Read internal metadata (if available)
        # item_date = datetime.strptime(src.tags()['TIFFTAG_DATETIME'], '%Y:%m:%d %H:%M:%S')

        # Fallback for this example
        item_date = datetime(2018, 1, 1) 

        # --- C. Create the Item ---
        # ID must be unique! Usually "Product_Tile_Date"
        item_id = os.path.splitext(os.path.basename(file_path))[0]
        
        item = pystac.Item(
            id=item_id,
            geometry=geometry,
            bbox=bbox,
            datetime=item_date,
            properties={
                "proj:epsg": proj_epsg,
                "proj:shape": src.shape,
                "proj:bbox": proj_bounds,
            }
        )

        # --- D. Add Asset ---
        item.add_asset(
            key="data",
            asset=pystac.Asset(
                href=file_path,
                title="Classification Data",
                media_type=pystac.MediaType.GEOTIFF,
                roles=["data"],
                extra_fields={
                    "eo:bands": [{"name": "classification"}]
                }
            )
        )
        
        items.append(item)